# Unified USBL Data Schema — Processed Output Visualisation

Visualises processed USBL data in the unified schema:
ship track, target positions, ship→target bearing arrows, and time-series
of depth, horizontal range, inclination angle, and uncertainty.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "qdch0ftq_20100428_020202"
SENSOR: str = "usbl_tracklink"  # "usbl_tracklink" or "usbl_evologics"

# NOTE: TrackLink deployments
# qdch0ftq_20100428_020202     → 201004_rv_linnaeus
# qdch0ftq_20110415_020103     → 201104_rv_naturaliste
# qdch0ftq_20120430_002423     → 201204_rv_linnaeus
# qdch0ftq_20130406_023610     → 201304_rv_linnaeus
# qdchdmy1_20110416_005411     → 201104_rv_naturaliste
# qdchdmy1_20120501_071203     → 201204_rv_linnaeus
# qdchdmy1_20130406_081713     → 201304_rv_linnaeus
# r23685bc_20100605_021022     → 201004_rv_linnaeus
# r23685bc_20120530_233021     → 201204_rv_linnaeus
# r23685bc_20140616_225022     → 201403_rv_linnaeus
# r29mrd5h_20090612_225306     → 200906_rv_challenger
# r29mrd5h_20110612_033752     → 201104_rv_naturaliste
# r29mrd5h_20130611_002419     → 201304_rv_linnaeus
# r7jjskxq_20101023_210332     → 201004_rv_linnaeus
# r7jjskxq_20121013_060425     → 201204_rv_linnaeus
# r7jjskxq_20131022_004934     → 201304_rv_linnaeus

# NOTE: Evologics deployments
# qd61g27j_20170523_040815     → 201705_silver_streak
# qdc5ghs3_20210315_230947     → 202102_poppag
# qdch0ftq_20170526_025746     → 201705_silver_streak
# qdch0ftq_20210315_034028     → 202102_poppag
# qdchdmy1_20170525_234624     → 201705_silver_streak
# qdchdmy1_20210315_081519     → 202102_poppag
# qtqxshxs_20150327_015552     → 201502_tasmania_bluefin
# qtqxshxs_20150328_000850     → 201502_tasmania_bluefin
# qtqxshxs_20150328_042551     → 201502_tasmania_bluefin

PROCESSED_DIR: Path = Path("/data/exos_01/acfr_messages_v4_telemetry_processed")
USBL_FILE: Path = PROCESSED_DIR / f"{DEPLOYMENT}_{SENSOR}.csv"

# Draw one arrow every ARROW_STEP pings to avoid clutter.
ARROW_STEP: int = 5

## Load data

In [ ]:
REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_roll",
    "ship_pitch",
    "ship_heading",
    "target_latitude",
    "target_longitude",
    "target_depth",
    "target_horizontal_range",
    "target_inclination_angle",
    "target_x",
    "target_y",
    "target_z",
    "horizontal_position_std",
    "depth_position_std",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)

missing: list[str] = [col for col in REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

# evologics_accuracy is present only in Evologics output.
has_evologics_accuracy: bool = "evologics_accuracy" in usbl.columns

print(f"Sensor       : {SENSOR}")
print(f"Deployment   : {DEPLOYMENT}")
print(f"Rows         : {len(usbl)}")
print(f"Evologics acc: {has_evologics_accuracy}")
usbl.head(3)

## Overview map: ship track and target positions

In [ ]:
t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
elapsed: np.ndarray = (t_s - t_s.min()) / 60.0

center_lat: float = float(usbl["target_latitude"].mean())
center_lon: float = float(usbl["target_longitude"].mean())

colorscale: str = "Viridis"

# Build customdata and hover template depending on sensor type.
if has_evologics_accuracy:
    customdata_cols: np.ndarray = np.stack(
        [
            usbl["target_depth"],
            usbl["target_horizontal_range"],
            usbl["evologics_accuracy"],
            usbl["horizontal_position_std"],
        ],
        axis=1,
    )
    target_hover: str = (
        "<b>USBL target</b><br>"
        "Lat: %{lat:.6f}<br>"
        "Lon: %{lon:.6f}<br>"
        "Depth: %{customdata[0]:.1f} m<br>"
        "Horizontal range: %{customdata[1]:.1f} m<br>"
        "Evologics accuracy: %{customdata[2]:.2f} m<br>"
        "Horizontal σ: %{customdata[3]:.2f} m<br>"
        "<extra></extra>"
    )
else:
    customdata_cols = np.stack(
        [
            usbl["target_depth"],
            usbl["target_horizontal_range"],
            usbl["horizontal_position_std"],
        ],
        axis=1,
    )
    target_hover = (
        "<b>USBL target</b><br>"
        "Lat: %{lat:.6f}<br>"
        "Lon: %{lon:.6f}<br>"
        "Depth: %{customdata[0]:.1f} m<br>"
        "Horizontal range: %{customdata[1]:.1f} m<br>"
        "Horizontal σ: %{customdata[2]:.2f} m<br>"
        "<extra></extra>"
    )

fig = go.Figure()

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Scattermapbox(
        lat=usbl["target_latitude"],
        lon=usbl["target_longitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=False,
        ),
        name="USBL target",
        customdata=customdata_cols,
        hovertemplate=target_hover,
    )
)

fig.update_layout(
    title=f"Ship track and USBL target positions — {DEPLOYMENT} ({SENSOR})",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Bearing arrows: ship → target

Each arrow is a line segment from the ship GPS position to the resolved target
position, drawn every `ARROW_STEP` pings to avoid clutter. The arrow direction
encodes the bearing from ship to target; length encodes horizontal range.

In [ ]:
sub: pd.DataFrame = usbl.iloc[::ARROW_STEP].reset_index(drop=True)
sub_elapsed: np.ndarray = elapsed[::ARROW_STEP]

# Build interleaved lat/lon arrays: ship → target → None for each ping.
n: int = len(sub)
arrow_lat: np.ndarray = np.empty(n * 3, dtype=object)
arrow_lon: np.ndarray = np.empty(n * 3, dtype=object)
arrow_lat[0::3] = sub["ship_latitude"].to_numpy()
arrow_lat[1::3] = sub["target_latitude"].to_numpy()
arrow_lat[2::3] = None
arrow_lon[0::3] = sub["ship_longitude"].to_numpy()
arrow_lon[1::3] = sub["target_longitude"].to_numpy()
arrow_lon[2::3] = None

fig_arrows = go.Figure()

fig_arrows.add_trace(
    go.Scattermapbox(
        lat=arrow_lat.tolist(),
        lon=arrow_lon.tolist(),
        mode="lines",
        line=dict(width=1, color="crimson"),
        name="Ship → target",
        hoverinfo="skip",
    )
)

fig_arrows.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines",
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hoverinfo="skip",
    )
)

fig_arrows.add_trace(
    go.Scattermapbox(
        lat=sub["target_latitude"],
        lon=sub["target_longitude"],
        mode="markers",
        marker=dict(
            size=5,
            color=sub_elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=float(elapsed.max()),
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        name="Target (subsampled)",
        customdata=np.stack(
            [sub["target_horizontal_range"], sub["target_inclination_angle"]],
            axis=1,
        ),
        hovertemplate=(
            "<b>Target</b><br>"
            "Horizontal range: %{customdata[0]:.1f} m<br>"
            "Inclination: %{customdata[1]:.1f}°<br>"
            "<extra></extra>"
        ),
    )
)

fig_arrows.update_layout(
    title=f"Ship → target bearing arrows — {DEPLOYMENT} ({SENSOR}, every {ARROW_STEP} pings)",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_arrows.show()

## Time series: depth, range, inclination, and uncertainty

In [ ]:
fig_ts = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Target depth (m)",
        "Horizontal range (m)",
        "Inclination angle (degrees)",
    ),
    vertical_spacing=0.08,
)

series: list[tuple[str, str, int]] = [
    ("target_depth", "steelblue", 1),
    ("target_horizontal_range", "darkorange", 2),
    ("target_inclination_angle", "seagreen", 3),
]

for col, color, row in series:
    fig_ts.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=usbl[col],
            mode="lines+markers",
            marker=dict(size=3),
            name=col,
            line=dict(color=color),
        ),
        row=row,
        col=1,
    )

fig_ts.update_yaxes(autorange="reversed", row=1, col=1)

fig_ts.update_layout(
    title=f"USBL time series — {DEPLOYMENT} ({SENSOR})",
    height=700,
    showlegend=False,
)

fig_ts.show()

## Time series: Vessel-Frame XYZ offsets

In [ ]:
fig_xyz = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Target X — Vessel-Frame forward (m)",
        "Target Y — Vessel-Frame starboard (m)",
        "Target Z — Vessel-Frame down (m)",
    ),
    vertical_spacing=0.07,
)

for row, (col, color) in enumerate(
    [
        ("target_x", "steelblue"),
        ("target_y", "darkorange"),
        ("target_z", "seagreen"),
    ],
    start=1,
):
    fig_xyz.add_trace(
        go.Scatter(
            x=usbl["timestamp"],
            y=usbl[col],
            mode="lines+markers",
            marker=dict(size=3),
            name=col,
            line=dict(color=color),
        ),
        row=row,
        col=1,
    )

fig_xyz.update_layout(
    title=f"Target Vessel-Frame XYZ offsets — {DEPLOYMENT} ({SENSOR})",
    height=700,
    showlegend=False,
)

fig_xyz.show()